In [1]:
import pandas as pd
import glob
import os

print("SENTRi-X Environment Active.")
print(f"Pandas Version: {pd.__version__}")

# 1. Define the relative path from the 'notebooks' folder to the ToN-IoT folder
ton_iot_path = "../data/raw/ton_iot/"

# 2. Use glob to automatically find all 23 CSV files in that folder
all_files = glob.glob(os.path.join(ton_iot_path, "*.csv"))
print(f"\nSuccessfully located {len(all_files)} ToN-IoT chunk files.")

# 3. Load ONLY the first chunk to prototype our ETL Pipeline safely
print(f"Loading {os.path.basename(all_files[0])} into Pandas...")
df = pd.read_csv(all_files[0])

print("\nExtraction Complete! Here is the shape of this single chunk:")
print(f"Total Rows (Network Flows): {df.shape[0]:,}")
print(f"Total Columns (Features): {df.shape[1]}")

# 4. Display the first 5 rows to verify the column headers
df.head()

SENTRi-X Environment Active.
Pandas Version: 3.0.1

Successfully located 23 ToN-IoT chunk files.
Loading Network_dataset_1.csv into Pandas...


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_25880\4210301357.py:17: DtypeWarning: Columns (0: src_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(all_files[0])



Extraction Complete! Here is the shape of this single chunk:
Total Rows (Network Flows): 1,000,000
Total Columns (Features): 46


,ts,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,1554198358,3.122.49.24,1883,192.168.1.152,52976,tcp,-,80549.530260,1762852,41933215,...,0,0,-,-,-,bad_TCP_checksum,-,F,0,normal
1,1554198358,192.168.1.79,47260,192.168.1.255,15600,udp,-,0.000000,0,0,...,0,0,-,-,-,-,-,-,0,normal
2,1554198359,192.168.1.152,1880,192.168.1.152,51782,tcp,-,0.000000,0,0,...,0,0,-,-,-,bad_TCP_checksum,-,F,0,normal
3,1554198359,192.168.1.152,34296,192.168.1.152,10502,tcp,-,0.000000,0,0,...,0,0,-,-,-,-,-,-,0,normal
4,1554198362,192.168.1.152,46608,192.168.1.190,53,udp,dns,0.000549,0,298,...,0,0,-,-,-,bad_UDP_checksum,-,F,0,normal


In [2]:
import numpy as np

print("--- Step 2: Data Cleaning & Feature Selection ---")

# 1. Fix the DtypeWarning (Mixed Types)
# We force Pandas to turn 'src_bytes' (and other math columns) into strict numbers. 
# The 'coerce' command tells it: "If you see a dash or a letter, turn it into a blank NaN."
numeric_columns = ['src_bytes', 'dst_bytes'] 
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
print("Successfully coerced mixed data types into strict numeric formats.")

# 2. Drop Overfitting Features
# We drop IPs and Timestamps so the AI learns the behavior of the attack, not the location.
drop_cols = ['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
print(f"Dropped contextual metadata. Features remaining: {df.shape[1]}")

# 3. Clean Missing and Infinite Values
# Replace any remaining string dashes '-' with mathematical NaN
df.replace('-', np.nan, inplace=True)
# Replace infinite network flow values with mathematical NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate how much data we drop
initial_rows = df.shape[0]
df.dropna(inplace=True)
final_rows = df.shape[0]

print(f"Purged {initial_rows - final_rows:,} corrupted rows (NaN/Inf).")
print(f"Cleaned Dataset Size: {final_rows:,} Rows")

df.head()

--- Step 2: Data Cleaning & Feature Selection ---
Successfully coerced mixed data types into strict numeric formats.
Dropped contextual metadata. Features remaining: 41
Purged 1,000,000 corrupted rows (NaN/Inf).
Cleaned Dataset Size: 0 Rows


,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,src_ip_bytes,dst_pkts,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type


In [3]:
import pandas as pd
import numpy as np

print("--- Step 2: Advanced Data Cleaning (Thresholding) ---")

# 1. We must reload the dataframe since the last script emptied it!
print("Reloading the dataset chunk...")
df = pd.read_csv(all_files[0], low_memory=False)

# 2. Fix the mixed data types
numeric_columns = ['src_bytes', 'dst_bytes'] 
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Drop Overfitting Features (IPs, Timestamps)
drop_cols = ['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

# 4. Standardize missing values
df.replace('-', np.nan, inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 5. Drop columns that are mostly empty BEFORE dropping rows
print("\nScanning for highly corrupted columns...")
threshold = 0.30 # If a column is missing more than 30% of its data, drop the column.
missing_percentages = df.isna().mean()
bad_columns = missing_percentages[missing_percentages > threshold].index.tolist()

if bad_columns:
    print(f"Dropping {len(bad_columns)} columns with >30% missing data:")
    print(bad_columns)
    df.drop(columns=bad_columns, inplace=True)
else:
    print("No columns exceeded the 30% missing data threshold.")

# 6. Now drop the remaining rows that have occasional NaNs
initial_rows = df.shape[0]
df.dropna(inplace=True)
final_rows = df.shape[0]

print(f"\nPurged {initial_rows - final_rows:,} corrupted rows.")
print(f"Cleaned Dataset Size: {final_rows:,} Rows")
print(f"Final Feature Count: {df.shape[1]} Columns")

df.head()

--- Step 2: Advanced Data Cleaning (Thresholding) ---
Reloading the dataset chunk...

Scanning for highly corrupted columns...
Dropping 23 columns with >30% missing data:
['service', 'dns_query', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_referrer', 'http_version', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice']

Purged 175 corrupted rows.
Cleaned Dataset Size: 999,825 Rows
Final Feature Count: 18 Columns


,proto,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,src_ip_bytes,dst_pkts,dst_ip_bytes,dns_qclass,dns_qtype,dns_rcode,http_request_body_len,http_response_body_len,http_status_code,label,type
0,tcp,80549.530260,1762852.0,41933215,OTH,0,252181,14911156,2,236,0,0,0,0,0,0,0,normal
1,udp,0.000000,0.0,0,S0,0,1,63,0,0,0,0,0,0,0,0,0,normal
2,tcp,0.000000,0.0,0,OTH,0,0,0,0,0,0,0,0,0,0,0,0,normal
3,tcp,0.000000,0.0,0,OTH,0,0,0,0,0,0,0,0,0,0,0,0,normal
4,udp,0.000549,0.0,298,SHR,0,0,0,2,354,0,0,0,0,0,0,0,normal


In [4]:
print("--- Step 3: Feature Inspection & Target Identification ---")

# 1. Display the surviving 18 columns and their data types
print("Surviving Features and Data Types:")
print(df.dtypes)

# 2. Check the distribution of the target variables
print("\n--- Threat Distribution ---")
if 'label' in df.columns:
    print("Binary Target ('label'):")
    # Multiplying by 100 gives us the exact percentage split
    print((df['label'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

if 'type' in df.columns:
    print("\nMulticlass Target ('type'):")
    print(df['type'].value_counts())
else:
    print("\nColumn 'type' not found. We will predict purely on binary 'label'.")

--- Step 3: Feature Inspection & Target Identification ---
Surviving Features and Data Types:
proto                         str
duration                  float64
src_bytes                 float64
dst_bytes                   int64
conn_state                    str
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
http_request_body_len       int64
http_response_body_len      int64
http_status_code            int64
label                       int64
type                          str
dtype: object

--- Threat Distribution ---
Binary Target ('label'):
label
1    79.15%
0    20.85%
Name: proportion, dtype: str

Multiclass Target ('type'):
type
scanning    791321
normal      208504
Name: count, dtype: int64


In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("--- Step 4: Feature Engineering & Scaling ---")

# 1. Isolate Features (X) and Target (y)
# We drop 'type' so the AI doesn't cheat by looking at the attack name.
X = df.drop(columns=['label', 'type'])
y = df['label'].astype(int)

print(f"Starting Feature Count: {X.shape[1]}")

# 2. One-Hot Encoding for categorical text
categorical_cols = ['proto', 'conn_state']
X = pd.get_dummies(X, columns=[col for col in categorical_cols if col in X.columns], drop_first=True)

# Convert all boolean columns generated by get_dummies to integers (0 or 1)
X = X.astype(float)

print(f"Feature Count after One-Hot Encoding: {X.shape[1]}")

# 3. Standardization (Scaling)
scaler = StandardScaler()

# Fit and transform the mathematical features
X_scaled = scaler.fit_transform(X)

# Convert back to a Pandas DataFrame for easy viewing
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\nTransformation Complete! The data is now a pure, normalized mathematical matrix.")
X_scaled_df.head()

--- Step 4: Feature Engineering & Scaling ---
Starting Feature Count: 16
Feature Count after One-Hot Encoding: 28

Transformation Complete! The data is now a pure, normalized mathematical matrix.


,duration,src_bytes,dst_bytes,missed_bytes,src_pkts,src_ip_bytes,dst_pkts,dst_ip_bytes,dns_qclass,dns_qtype,...,conn_state_RSTOS0,conn_state_RSTR,conn_state_RSTRH,conn_state_S0,conn_state_S1,conn_state_S2,conn_state_S3,conn_state_SF,conn_state_SH,conn_state_SHR
0,437.189979,6.520803,129.846877,-0.00465,745.450959,168.475085,-0.003013,-0.004218,-0.04259,-0.02478,...,-0.226216,-0.013269,-0.014529,-1.320765,-0.022323,-0.005001,-0.007072,-0.11209,-0.006709,-0.147038
1,-0.014199,-0.004773,-0.006051,-0.00465,-0.005646,-0.005226,-0.006616,-0.005042,-0.04259,-0.02478,...,-0.226216,-0.013269,-0.014529,0.757137,-0.022323,-0.005001,-0.007072,-0.11209,-0.006709,-0.147038
2,-0.014199,-0.004773,-0.006051,-0.00465,-0.008602,-0.005938,-0.006616,-0.005042,-0.04259,-0.02478,...,-0.226216,-0.013269,-0.014529,-1.320765,-0.022323,-0.005001,-0.007072,-0.11209,-0.006709,-0.147038
3,-0.014199,-0.004773,-0.006051,-0.00465,-0.008602,-0.005938,-0.006616,-0.005042,-0.04259,-0.02478,...,-0.226216,-0.013269,-0.014529,-1.320765,-0.022323,-0.005001,-0.007072,-0.11209,-0.006709,-0.147038
4,-0.014196,-0.004773,-0.005128,-0.00465,-0.008602,-0.005938,-0.003013,-0.003805,-0.04259,-0.02478,...,-0.226216,-0.013269,-0.014529,-1.320765,-0.022323,-0.005001,-0.007072,-0.11209,-0.006709,6.800952


In [6]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("--- Step 5: Data Splitting & SMOTE Balancing ---")

# 1. Split the data FIRST (80% Training, 20% Testing)
# 'stratify=y' ensures the 79/21 ratio is perfectly maintained in both splits before we alter it.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Original Training Set: {X_train.shape[0]:,} rows")
print(f"Original Testing Set:  {X_test.shape[0]:,} rows")

print("\n--- Before SMOTE (Training Set Imbalance) ---")
print((y_train.value_counts(normalize=True) * 100).round(2).astype(str) + '%')

# 2. Apply SMOTE to the training set ONLY
print("\nApplying SMOTE... (Injecting synthetic minority data)")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. Verify the new, perfectly balanced training environment
print("\n--- After SMOTE (New Training Set Balance) ---")
print((y_train_smote.value_counts(normalize=True) * 100).round(2).astype(str) + '%')
print(f"New Training Set Size: {X_train_smote.shape[0]:,} rows")

--- Step 5: Data Splitting & SMOTE Balancing ---
Original Training Set: 799,860 rows
Original Testing Set:  199,965 rows

--- Before SMOTE (Training Set Imbalance) ---
label
1    79.15%
0    20.85%
Name: proportion, dtype: str

Applying SMOTE... (Injecting synthetic minority data)

--- After SMOTE (New Training Set Balance) ---
label
0    50.0%
1    50.0%
Name: proportion, dtype: str
New Training Set Size: 1,266,114 rows


In [7]:
import joblib
import os

print("--- Step 6: Exporting the Processed ML Matrices (ToN-IoT) ---")

processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)

# Appending 'ton_iot_' to the filenames to protect them from being overwritten later
print("Exporting ToN-IoT matrices to disk...")

joblib.dump(X_train_smote, os.path.join(processed_dir, "ton_iot_X_train_smote.pkl"))
joblib.dump(y_train_smote, os.path.join(processed_dir, "ton_iot_y_train_smote.pkl"))
joblib.dump(X_test, os.path.join(processed_dir, "ton_iot_X_test.pkl"))
joblib.dump(y_test, os.path.join(processed_dir, "ton_iot_y_test.pkl"))

joblib.dump(scaler, os.path.join(processed_dir, "ton_iot_scaler.pkl"))

print(f"\nPipeline Complete! ToN-IoT artifacts saved securely to: {processed_dir}")

--- Step 6: Exporting the Processed ML Matrices (ToN-IoT) ---
Exporting ToN-IoT matrices to disk...

Pipeline Complete! ToN-IoT artifacts saved securely to: ../data/processed/
